# Finalize: all data in one collection (2004-2026.09)

**notebook version**: `3` (27 Oct 2026)

## ℹ️ About this notebook

This notebook collects all data available so far into one file, on one 30-minute timestamp from
the start of the meteo record in 2004 to 30 September 2026. The datasets of a release are cut from
it by period, for example FP2026.1 (2016 to September 2026) or a single setup period such as
`2026_1`:

```python
finalize.loc['2026-01-01 00:15':'2026-09-30 23:45']
```

It collects three kinds of data:

| Source | What it adds | Period |
|---|---|---|
| `20_L1_FLUXES/IRGA72/99_IRGA72_L1_MERGED_2016-2026.09` | Level-1 fluxes: the EddyPro output of the 18 `IRGA72` runs | 11 January 2016 to September 2026 |
| `10_METEO/30_PRODUCTS/99_METEO_MERGED_2004-2026.09` | all meteo: every product that `99` merges | 2004 to September 2026 (`08` to `13` to December 2025) |
| `30_FLUX_POST-PROCESSING_CHAIN/IRGA72/99_FPPC_IRGA72_MERGED_2016-2026.09` | the flux post-processing of `NEE`, `LE` (with `ET`) and `H` | 2016 to September 2026 |

So far the Level-1 fluxes and their post-processing are those of the enclosed-path LI-7200
(`IRGA72`) only. The fluxes of the open-path LI-7500 (`IRGA75`, 2004 to 2017 and 2019) join once
their chain has been run.

The meteo comes from `99` only: `99` collects the meteo products, and a product reaches this file by
being merged there. `99` merges all thirteen products. `01` to `07` reach September 2026; `08` to
`13` (`PREC`, `SWC`, `TS`, `G`, `SW_OUT`, `LW_OUT`) still end in December 2025 and are empty from
January 2026 until they are extended.

Each source adds only the columns the file does not have yet. A column that two sources share must
hold the same values in both. `22.1` carries `SW_IN_POT` from EddyPro, which runs about 15 minutes
ahead of the timestamp, as if it were computed for the end of each half-hour. The flux notebooks
compute their own `SW_IN_POT` for the middle of each half-hour, from which their day and night
flags come. The collection keeps that one as `SW_IN_POT` and the EddyPro one as
`SW_IN_POT_EDDYPRO`.

The last section lists every variable of the collection with its unit, a short description, its
source and its period. The units and descriptions come from `variable_info.py` in this folder.

## ⚙️ Settings

In [1]:
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
FILES = {
    'L1 fluxes IRGA72': rf"{DATADIR}\20_L1_FLUXES\IRGA72\99_IRGA72_L1_MERGED_2016-2026.09.parquet",
    'meteo': rf"{DATADIR}\10_METEO\30_PRODUCTS\99_METEO_MERGED_2004-2026.09.parquet",
    'FPPC IRGA72': (rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA72"
                    r"\99_FPPC_IRGA72_MERGED_2016-2026.09.parquet"),
}
PERIOD_START, PERIOD_END = '2004-01-01 00:15', '2026-09-30 23:45'

OUTNAME = "10_FLUXES_and_METEO_2004-2026.09"  # same name as this notebook
OUTPATH = rf"{DATADIR}\80_FINALIZE"

In [2]:
from datetime import datetime
import importlib.metadata

import diive as dv
import numpy as np
import pandas as pd

from variable_info import describe_all, info_section

NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-10-27 13:00:39
diive version: v0.93.0


## 📂 Load data

In [3]:
frames = {k: dv.load_parquet(filepath=v) for k, v in FILES.items()}
for name, frame in frames.items():
    assert frame.index.is_unique, f"{name} has duplicate timestamps"
    print(f"{name:>16}: {frame.shape[0]:,} records x {frame.shape[1]:>3} columns, "
          f"{frame.index[0]} -> {frame.index[-1]}")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\20_L1_FLUXES\IRGA
72\99_IRGA72_L1_MERGED_2016-2026.09.parquet (0.253 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\99_METEO_MERGED_2004-2026.09.parquet (0.071 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA72\99_FPPC_IRGA72_MERGED_2016-2026.09.parquet (0.091 seconds).

L1 fluxes IRGA72: 187,937 records x 329 columns, 2016-01-11 15:45:00 -> 2026-09-30 23:45:00
           meteo: 398,832 records x  61 columns, 2004-01-01 00:15:00 -> 2026-09-30 23:45:00
     FPPC IRGA72: 188,448 records x 158 columns, 2016-01-01 00:15:00 -> 2026-09-30 23:45:00


## Collect

In [4]:
def add_new_columns(main, frame, name, source):
    """Add the columns of frame that main does not have yet, on main's timestamps.

    A column that main already has must hold the same values in frame, so that no result is
    dropped silently. Returns the extended frame and records each new column's source.
    """
    assert frame.index.isin(main.index).all(), f"{name} has timestamps outside the main index"
    frame = frame.reindex(main.index)
    new = [c for c in frame.columns if c not in main.columns]
    for c in frame.columns.difference(new):
        a, b = main[c], frame[c]
        both = a.notna() & b.notna()
        assert np.allclose(a[both], b[both], equal_nan=True), f"{c} differs between main and {name}"
    for c in new:
        source[c] = name
    print(f"{name}: {len(new)} new columns, {frame.shape[1] - len(new)} already present")
    return pd.concat([main, frame[new]], axis=1)

In [5]:
# EddyPro's potential radiation, renamed so the one of the flux chain takes its name.
frames['L1 fluxes IRGA72'] = frames['L1 fluxes IRGA72'].rename(columns={'SW_IN_POT': 'SW_IN_POT_EDDYPRO'})
index = pd.date_range(PERIOD_START, PERIOD_END, freq='30min', name='TIMESTAMP_MIDDLE')
finalize = pd.DataFrame(index=index)
SOURCE = {}
for name, frame in frames.items():
    finalize = add_new_columns(finalize, frame, name, SOURCE)
assert finalize.columns.is_unique
fullset = finalize  # name used by the export cell
DESCRIBED = describe_all(finalize.columns)  # unit and description of every column
print(f"Finalize: {finalize.shape[0]:,} records x {finalize.shape[1]} columns, "
      f"{finalize.memory_usage().sum() / 1e9:.1f} GB in memory")

L1 fluxes IRGA72: 329 new columns, 0 already present
meteo: 61 new columns, 0 already present
FPPC IRGA72: 158 new columns, 0 already present
Finalize: 398,832 records x 548 columns, 1.8 GB in memory


### 📋 Table: columns per source

In [6]:
display(pd.Series(SOURCE).value_counts().reindex(list(FILES)).rename('columns').to_frame())

,columns
L1 fluxes IRGA72,329
meteo,61
FPPC IRGA72,158


### 📋 Table: coverage per source and year

The share of records per year in which a source has at least one value. A source that ends before
30 September 2026 shows 0 in the years after its end.

In [7]:
_cov = {}
for _name in FILES:
    _cols = [c for c, s in SOURCE.items() if s == _name]
    _cov[_name] = finalize[_cols].notna().any(axis=1).groupby(finalize.index.year).mean() * 100
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame(_cov).round(0).astype(int).rename_axis('year'))

,L1 fluxes IRGA72,meteo,FPPC IRGA72
year,,,
2004,0,100,0
2005,0,100,0
2006,0,100,0
2007,0,100,0
2008,0,100,0
2009,0,100,0
2010,0,100,0
2011,0,100,0
2012,0,100,0


## 💾 Export

The collection as parquet, and next to it an `_INFO.md` file that lists every column with its unit and a short description, per source.

In [8]:
filepath = dv.save_parquet(data=finalize, filename=OUTNAME, outpath=OUTPATH)

_lines = [f"# {OUTNAME}", "",
          f"{fullset.index[0]} to {fullset.index[-1]} ({fullset.index.name}, UTC+01:00), "
          f"{fullset.shape[0]:,} records x {fullset.shape[1]} columns.",
          f"Written {datetime.now():%Y-%m-%d %H:%M} by `{OUTNAME}.ipynb`.", "",
          "| source | file |", "|---|---|"]
_lines += [f"| {k} | `{v}` |" for k, v in FILES.items()]
for _name in FILES:
    _lines += info_section(_name, [c for c, s in SOURCE.items() if s == _name], DESCRIBED)
with open(rf"{OUTPATH}\{OUTNAME}_INFO.md", "w", encoding="utf-8") as _f:
    _f.write("\n".join(_lines) + "\n")
print(f"Wrote {OUTPATH}\\{OUTNAME}_INFO.md")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\10_FL
UXES_and_METEO_2004-2026.09.parquet (4.993 seconds).

Wrote F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\10_FLUXES_and_METEO_2004-2026.09_INFO.md


## 📋 Variables in the collection

Every variable of the collection, in the order of the file: its unit, a short description, its
source, the number of records with a value, and the first and last of these records.

### 📋 Table: all variables

In [9]:
_valid = finalize.notna()
VARIABLES = pd.DataFrame({
    'unit': pd.Series({c: u for c, (u, d) in DESCRIBED.items()}),
    'description': pd.Series({c: d for c, (u, d) in DESCRIBED.items()}),
    'source': pd.Series(SOURCE),
    'records': _valid.sum(),
    'first': finalize.apply(lambda s: s.first_valid_index()),
    'last': finalize.apply(lambda s: s.last_valid_index()),
}).reindex(finalize.columns)
VARIABLES.index.name = 'variable'
VARIABLES.insert(0, '#', range(1, len(VARIABLES) + 1))
print(f"{len(VARIABLES)} variables; without any value: {int((VARIABLES['records'] == 0).sum())}")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(VARIABLES)

548 variables; without any value: 0


,#,unit,description,source,records,first,last
variable,,,,,,,
AIR_CP,1,J kg-1 K-1,Specific heat of moist air at constant pressure,L1 fluxes IRGA72,179251,2016-01-11 15:45:00,2026-09-30 23:45:00
AIR_DENSITY,2,kg m-3,Density of moist air,L1 fluxes IRGA72,179251,2016-01-11 15:45:00,2026-09-30 23:45:00
AIR_MV,3,m3 mol-1,Molar volume of moist air,L1 fluxes IRGA72,179251,2016-01-11 15:45:00,2026-09-30 23:45:00
AIR_RHO_CP,4,J m-3 K-1,Volumetric heat capacity of moist air,L1 fluxes IRGA72,179251,2016-01-11 15:45:00,2026-09-30 23:45:00
AOA_METHOD,5,-,Angle-of-attack correction (EddyPro code),L1 fluxes IRGA72,179251,2016-01-11 15:45:00,2026-09-30 23:45:00
AXES_ROTATION_METHOD,6,-,Coordinate rotation method (EddyPro code),L1 fluxes IRGA72,179251,2016-01-11 15:45:00,2026-09-30 23:45:00
BADM_HEIGHTC,7,m,Canopy height,L1 fluxes IRGA72,179251,2016-01-11 15:45:00,2026-09-30 23:45:00
BADM_INSTPAIR_EASTWARD_SEP_GA_CO2,8,cm,"Eastward separation of the analyser from the sonic, CO2",L1 fluxes IRGA72,174863,2016-01-22 12:15:00,2026-09-30 23:45:00
BADM_INSTPAIR_EASTWARD_SEP_GA_H2O,9,cm,"Eastward separation of the analyser from the sonic, H2O",L1 fluxes IRGA72,174988,2016-01-12 17:15:00,2026-09-30 23:45:00


***

# End of notebook.

In [10]:
NOTEBOOK_END = datetime.now()
print(f"Runtime: {NOTEBOOK_END - NOTEBOOK_START}")

Runtime: 0:00:07.636463
